# Dynex G70

In [19]:
import networkx as nx

from pyqubo import Array
import pickle
from dynex import DynexSampler, BQM, DynexConfig, ComputeBackend, QPUModel

config = DynexConfig(compute_backend=ComputeBackend.QPU, qpu_model=QPUModel.APOLLO_RC1, use_notebook_output=True)

In [20]:
filename = 'G70.dat' 

## Load Graph from File

In [21]:
# build graph from file:
G = nx.Graph()
f = open(filename, "r").readlines()
for ele in f[1:]:
    i, j, v = ele.split()
    G.add_edges_from([(int(i),int(j))])

## Construct QUBO

In [22]:
n = 10001
X = Array.create('X', n, vartype='SPIN')
H0 = 0
for (u, v) in G.edges:
    H0 -= (1 - X[u] * X[v]) / 2.0

In [23]:
model = H0.compile()

In [24]:
bqm = model.to_bqm()
factor = bqm.normalize()
print('bqm was normalized with factor', factor)
model = BQM(bqm)

bqm was normalized with factor 0.1111111111111111


## Sample Sweep

In [25]:
best_cut = 0
best_sol = []

In [26]:
all_sol = []

sampler = DynexSampler(model, config=config)

for i in range(1, 10):
    # Sample on Dynex
    print('Sweep', i, '...')
    sampleset = sampler.sample(num_reads=10, annealing_time=100)
    e = sampleset.first.energy / factor
    if e < best_cut:
        best_cut = e
        best_sol = sampleset.first
        print('IMPROVED CUT FOUND:', best_cut)
        # save solution locally:
        with open('maxcut_best_' + str(round(abs(e))) + '.pkl', 'wb') as outp:
            pickle.dump(best_sol.sample, outp)
    all_sol.append(e)
    print('Sweep', i, 'cut:', e, 'best cut:', best_cut, 'all:', all_sol)

INFO: [DYNEX-APOLLO-RC1] Timing breakdown:
INFO: [DYNEX-APOLLO-RC1]   Job upload:        0.36s
INFO: [DYNEX-APOLLO-RC1]   Time to 1st shot:  6.58s
INFO: [DYNEX-APOLLO-RC1]   Compute (Apollo):  6.58s
INFO: [DYNEX-APOLLO-RC1]   Solution download: 0.05s
INFO: [DYNEX-APOLLO-RC1]   Total elapsed:     6.98s
INFO: [DYNEX-APOLLO-RC1] Finished read after 3.397299999998893 seconds
INFO: [DYNEX-APOLLO-RC1] Sampleset ready with energy   X[10000] X[1000] X[1001] X[1002] X[1003] X[1008] ... X[99]  energy num_oc.
0        1       0       0       1       0       0 ...     1 -1057.0       1
['BINARY', 1 rows, 1 samples, 8646 variables]


Sweep 9 cut: -9512.999999999574 best cut: -9527.99999999955 all: [np.float64(-9514.99999999958), np.float64(-9525.99999999958), np.float64(-9526.99999999955), np.float64(-9522.999999999573), np.float64(-9527.99999999955), np.float64(-9517.999999999576), np.float64(-9515.99999999958), np.float64(-9514.999999999573), np.float64(-9512.999999999574)]


In [27]:
# load best solution:
with open('maxcut_best_9553.pkl', 'rb') as inp:
    x = pickle.load(inp)

In [28]:
# Validate cut result:
lut = x

# Interpret best result in terms of nodes and edges
S0 = [node for node in G.nodes if not lut['X[' + str(node) + ']']]
S1 = [node for node in G.nodes if lut['X[' + str(node) + ']']]
cut_edges = [(u, v) for u, v in G.edges if lut['X[' + str(u) + ']'] != lut['X[' + str(v) + ']']]
uncut_edges = [(u, v) for u, v in G.edges if lut['X[' + str(u) + ']'] == lut['X[' + str(v) + ']']]

print('Maxcut result:', len(cut_edges))

Maxcut result: 9553
